## Pre-trained GPT 모델 Fine-tuning 하기
> Fine-tuning: 기존 LLM에게 우리 회사의 매뉴얼이나 특정한 말투를 집중적으로 가르쳐서 전문가로 만드는 과정
>https://developers.openai.com/api/reference/python/resources/fine_tuning/subresources/jobs/methods/create
- fine-tuning에 필요한 데이터셋은 Chat API와 동일한 형식의 **대화**형태 여야하며,
- 각 메시지에 role별로 content가 들어간 메시지 목록이어야 함 (system, user, assistant)
- OpenAI의 fine-tuning API는 최소 10개 이상의 대화 데이터가 필요하며, 비용상 50개 미만의 소량 데이터셋으로 학습시켜 결과 먼저 확인 -> 필요시 추가
- OpenAI공식 페이지에서는 100개 미만의 데이터를 소량의 데이터로 정의하고 있음
- fine-tuning 전에 모델에 가장 잘 맞는 System Prompt를 설계한 뒤 해당 텍스트가 포함된 학습 데이터를 제작하는 것이 좋음
- 일부 학습 데이터에는 기존 모델이 **원하는대로 작동하지 않았을 경우** 제대로 된 응답 데이터를 포함시켜야 성능 향상
    - **원하는대로 작동하지 않았던 응답** Q. 프랑스 수도 어디야? A. 프랑스는 세계에서 가장 유명한 관광도시예요!
    - **fine-tuning에 넣어줄 제대로 된 데이터** Q. 프랑스 수도 어디야? A. 프랑스 수도는 파리입니다.

In [ ]:
# openAI 모델들이 사용하는 base토큰화 및 인코딩 지원 모듈 -> 텍스트 토크나이저
!pip install tiktoken

In [2]:
import os
import time
import json
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from datetime import datetime, UTC
import tiktoken
# 딕셔너리에 존재하지 않는 key에 대한 접근시, 자료형에 따라 디폴트 값을 생성해주는 클래스
from collections import defaultdict

In [3]:
load_dotenv()
MY_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=MY_API_KEY)

## **1. Fine-tuning 전 데이터 로드 및 검증 단계**
- 비용이 많이 들 수 있기 때문에 사전에 적합한 데이터 형태인지를 먼저 검증하고 진행하기
- OpenAI의 fine-tuning API 사용시 데이터 구조는 `JSONL`이어야 함
- `JSONL(JSON Lines)`: 일반 JSON이 전체 데이터를 하나의 객체로 보는 것에 비해, JSONL은 한줄당 JSON객체가 존재하는 형태.
- 한 줄씩 또는 청크 단위로 읽고 쓰는 것에 효율적. 대규모 데이터 세트를 구성하는데 적합

#### 1) 데이터를 직접 JSONL 파일로 저장하기

In [4]:
data = [
    {"messages": [
        {"role":"system", "content": "You are a chatbot that gives clear answers."},
        {"role":"user", "content":"What can i do?"},
        {"role":"assistant", "conetent": "Ask a question about the part you want."}
    ]},
    {"messages": [
        {"role":"system", "content": "You are a chatbot that gives clear answers."},
        {"role":"user", "content":"What can i do?"},
        {"role":"assistant", "conetent": "Ask a question about the part you want."}
    ]}
]

with open('data/my_dataset.jsonl', 'w', encoding='utf-8') as f:
    # json파일 저장시 한 줄에 json객체(딕셔너리)를 하나씩 저장해주는 방식
    for i in data :
        f.write(json.dumps(i) + '\n')

#### 2) OpenAI 샘플용 JSONL 데이터 활용
- toy_chat_fine_tuning.jsonl

In [56]:
data_path = 'data/toy_chat_fine_tuning.jsonl'

with open(data_path, 'r', encoding='utf-8') as f :
    # JSONL 불러올 때 내부의 JSON 객체들을 하나씩 불러와야 함
    dataset = [json.loads(line) for line in f]

print('샘플 수', len(dataset))
print()
dataset

샘플 수 5



[{'messages': [{'role': 'system',
    'content': 'You are a happy assistant that puts a positive spin on everything.'},
   {'role': 'user', 'content': 'I fell off my bike today.'},
   {'role': 'assistant',
    'content': "It's great that you're getting exercise outdoors!"}]},
 {'messages': [{'role': 'system',
    'content': 'You are a happy assistant that puts a positive spin on everything.'},
   {'role': 'user', 'content': 'I lost my tennis match today.'},
   {'role': 'assistant', 'content': "It's ok, it happens to everyone."},
   {'role': 'user', 'content': 'But I trained so hard!'},
   {'role': 'assistant', 'content': 'It will pay off next time.'},
   {'role': 'user', 'content': "I'm going to switch to golf."},
   {'role': 'assistant', 'content': 'Golf is fun too!'},
   {'role': 'user', 'content': "I don't even know how to play golf."},
   {'role': 'assistant', 'content': "It's easy to learn!"}]},
 {'messages': [{'role': 'user', 'content': 'I lost my book today.'},
   {'role': 'assi

In [6]:
dataset[0]

{'messages': [{'role': 'system',
   'content': 'You are a happy assistant that puts a positive spin on everything.'},
  {'role': 'user', 'content': 'I fell off my bike today.'},
  {'role': 'assistant',
   'content': "It's great that you're getting exercise outdoors!"}]}

In [7]:
dataset[0]['messages']

[{'role': 'system',
  'content': 'You are a happy assistant that puts a positive spin on everything.'},
 {'role': 'user', 'content': 'I fell off my bike today.'},
 {'role': 'assistant',
  'content': "It's great that you're getting exercise outdoors!"}]

#### 3) 실제 fine-tuning에 사용할 시사 상식 데이터 (10개)

In [8]:
data_path = 'data/common_sense_train.jsonl'

with open(data_path, 'r', encoding='utf-8') as f :
    dataset = [json.loads(line) for line in f]

print('샘플 수', len(dataset))
print()
dataset

샘플 수 10



[{'messages': [{'role': 'system', 'content': '너는 지식이 풍부하지만 시크하게 응답하는 챗봇이야.'},
   {'role': 'user', 'content': "제주도는 '삼다도'라고 불리기도 하는데 '삼다'에 해당되는 것은 무엇일까요?"},
   {'role': 'assistant',
    'content': "정답은 '여자', '바람', '돌' 이고 이 세가지가 많다고 해서 '삼다' 라고 불림."}]},
 {'messages': [{'role': 'system', 'content': '너는 지식이 풍부하지만 시크하게 응답하는 챗봇이야.'},
   {'role': 'user',
    'content': '뮤지컬, 연극, 오페라, 음악회 등의 공연이 끝난 후에 관객이 박수를 보내 배우들을 다시 무대로 나오게 하는 것을 무엇이라 할까요?'},
   {'role': 'assistant',
    'content': "정답은 '커튼콜'이며 '앵콜'과는 달리 추가 무대를 요청하는 것이 아니라 단순히 등장을 요구하는 것."}]},
 {'messages': [{'role': 'system', 'content': '너는 지식이 풍부하지만 시크하게 응답하는 챗봇이야.'},
   {'role': 'user', 'content': '프랑스의 수도는 어디인가요?'},
   {'role': 'assistant',
    'content': '프랑스의 수도는 파리임. 파리는 예술의 도시라고 불리며 세계 최고의 관광 도시 중 하나임.'}]},
 {'messages': [{'role': 'system', 'content': '너는 지식이 풍부하지만 시크하게 응답하는 챗봇이야.'},
   {'role': 'user', 'content': '세계에서 가장 큰 바다는 무엇인가요?'},
   {'role': 'assistant',
    'content': '세계에서 가장 큰 바다는 태평양임. 태평양은 오대양의 하나로 지구 표면의 1/3을 차지하며 표면적

### **데이터 사전 검증 진행**
- OpenAI 공식 검증 코드 : https://cookbook.openai.com/examples/chat_finetuning_data_prep

#### 1) 데이터셋이 OpenAI의 fine-tuning API에서 기대하는 형식을 준수하는지 검증

In [ ]:
# defaultdict 예시
format_errors = defaultdict(int)
format_errors

defaultdict(int, {})

In [10]:
format_errors['test']
format_errors

# int로 설정할 경우 정수형의 기본값인 0이 들어가 동작함

defaultdict(int, {'test': 0})

In [11]:
format_errors['test2'] += 1
format_errors

defaultdict(int, {'test': 0, 'test2': 1})

In [58]:
format_errors = defaultdict(int)

for ex in dataset :          # dataset은 list형태이며, ex는 하나의 messages로 dict 형태
    # <ex가 dict타입이 아니라면 카운트>
     # isinstance: 어떤 클래스의 객체인지 확인(객체명, 클래스명) -> T, F로 반환
    if not isinstance(ex, dict) :
        # ex가 dict가 아니라면 'data_type'의 value를 1 증가
        format_errors['data_type'] += 1
        continue

    # <messages의 value 값이 list가 아니라면 카운트>
     # ex에서 'messages'라는 key가 있으면 해당 value값을 출력하고 없으면 None 반환
    messages = ex.get('messages', None)
    if not messages :
        format_errors['missing_messages_list'] += 1
        continue

    # messages(리스트) 내부의 각 딕셔너리에 접근
    for message in messages : 
        # <key값에 'role'과 'content'가 포함되어 있지 않다면 카운트>
        if 'role' not in message or 'content' not in message :
            format_errors['message_missing_key'] += 1

        # <모든 message의 key값에서 아래 5개 외에 다른 값이 있다면 카운트>
         # 채팅 API외 모든 API들이 가질 수 있는 key의 종륜느 아래 5가지
          # any... 괄호 안의 조건 중 하나라도 True인 경우 True 반환
        if any(key not in ('role', 'content', 'name', 'function_call', 'weight') for key in message):
            format_errors['message_unrecognized_key'] += 1

        # <message의 role이 아래 4개 중 하나가 아니거나, 값이 없다면 카운트>
         # ROLE값이 될 수 있는 것은 아래 4개
        if message.get('role', None) not in ('system', 'user', 'assistant', 'function') :
            format_errors['message_unrecognized_role'] += 1

        # <message의 content에 value값이 존재하지 않거나, content가 문자열이 아니라면 카운트>
         # content에는 질의나 응답이 들어가야하므로 문자열이어야 함
        content = message.get('content', None)
        if not content or not isinstance(content, str):
            format_errors['missing_content'] += 1

    # <각 message의 role에 assistant의 응답이 하나라도 없다면 카운트>
     # message들 중 assistant의 응답이 없으면 False (앞에 not이 붙으므로 조건문은 True)
    if not any(message.get('role', None)=='assistant' for message in messages) :
        format_errors['example_missing_assistant_message'] += 1


# 발생한 에러가 있다면 (format_errors에 값이 있다면)
if format_errors : 
    print('에러 발견 :')
    for key, value in format_errors.items() :
        print(f'{key}: {value}')
else :
    print('에러가 없습니다.')

에러가 없습니다.


#### 2) 누락된 메시지 식별, 메시지와 토큰 수 확인
-  `tiktoken`: 텍스트 토크나이저. 비용 계산, 입력 제한 확인.
- "이 대화가 에러 없이 돌아갈까?", "돈은 얼마나 나올까?", "문서를 어떻게 잘라야 모델이 잘 알아먹을까?"라는 실무적인 문제를 해결하기 위한 **'측정기'**

In [13]:
# tiktoken 예시
 # get_encoding : 특정 토크나이저 인코딩 모델 검색
 # 'o200k_base' : gpt-4o 및 gpt-4o-mini 모델이 사용하는 토크나이저 인코딩 모델
 # 'cl100k_base' : gpt-3.5 외 기타 모델이 사용하는 토크나이저 인코딩 모델
encoding = tiktoken.get_encoding('o200k_base')

print(encoding.encode('hello my name is minjung'))

[24912, 922, 1308, 382, 1349, 121775]


In [14]:
encoding = tiktoken.get_encoding('o200k_base')

# <메시지 목록의 총 토큰 수 계산 함수>
 # tokens_per_message : 메시지 내 'role', 'content', '{}' 요소들에 대한 토큰 개수
 #                      (대략 한 요소당 토큰 1개로 총 3개의 토큰으로 보내겠다는 의미)
 # tokens_per_name : 모델명이 존재할 때 대략적으로 토큰 1개 추가
 #                  (사용자가 assistant의 이름을 지정할 경우 name이 생성됨)
def num_tokens_from_messages(messages, tokens_per_message=3, tokens_per_name=1) : 
    num_tokens = 0
    for message in messages :
        # message별로 'role', 'content', '{}'가 있을 것이므로 토큰 3개 추가
        num_tokens += tokens_per_message

        for key, value in message.items() :
            # value값('role' 및 'content'의 value)이 토큰화 + 인코딩된 길이를 num_tokens에 더해줌
            num_tokens += len(encoding.encode(value))

            # key에 name이 있다면 (assistant의 이름을 설정한 경우 생김)
            if key == 'name' :
                num_tokens += tokens_per_name

    # 전체 대화를 표시하거나 구분할 수 있는 토큰들을 대략 3개 추가
     # 1) 하나의 messages 전체를 감싸는 '{}'
     # 2) 한 message의 끝을 표시하는 '<SEP>' (특수토큰)
     # 3) 각각의 message를 구분해주는 ','
    num_tokens += 3
    return num_tokens


# <assistant가 응답한 메시지 총 토큰 수>
# (모델의 max_output_tokens 기준에 맞는지 판단하기 위해)
def num_assistant_tokens_from_messages(messages) :
    num_tokens = 0
    for message in messages :
        # 모델의 응답인 경우 (role이 assistant인 경우)
        if message['role']=='assistant' :
            # 'content'의 value값만 토큰화+인코딩하여 개수에 추가
            num_tokens += len(encoding.encode(message['content']))
    return num_tokens


# <message의 토큰 길이에 대한 통계 정보 출력>
def print_statistics(values) :
    print(f'min / max: {min(values)}, {max(values)}')
    print(f'mean / median: {np.mean(values)}, {np.median(values)}')

In [15]:
max_output_tokens = 16384      # 모델 최대 출력 토큰 수 (gpt-4o-mini)
n_missing_system = 0           # role에 'syste'이 없는 경우의 대화를 카운트
n_missing_user = 0           # role에 'user'가 없는 경우의 대화를 카운트
n_messages = []              # 각 대화의 message개수가 담길 빈 리스트
total_tokens_lens = []            # 각 대화의 총 토큰 수가 담길 빈 리스트
assistant_message_lens = []           # assistant가 보낸 메시지의 길이가 담길 빈 리스트

for ex in dataset :
    messages = ex['messages']

    # 각 message에서 role이 system인 경우가 하나도 없다면 카운트
    if not any(message['role']=='system' for message in messages) :
        n_missing_system += 1
    if not any(message['role']=='user' for message in messages) :
        n_missing_user += 1
    
    #각 messages의 message 개수를 리스트에 추가
    n_messages.append(len(messages))
    # 메시지 목록의 총 토큰 개수 함수 실행 후 리스트에 추가
    total_tokens_lens.append(num_tokens_from_messages(messages))
    # 모델 출력 토큰 수 함수 실행 후 리스트에 추가
    assistant_message_lens.append(num_assistant_tokens_from_messages(messages))

print('system 메시지 누락 수', n_missing_system)
print('user 메시지 누락 수', n_missing_user)
print()
print('### 대화당 메시지 수 통계:')
print_statistics(n_messages)   # 통계치 함수 실행
print()
print('### 대화당 전체 토큰 수 통계:')
print_statistics(total_tokens_lens)
print()
print('### 대화당 모델 출력 토큰 수 통계:')
print_statistics(assistant_message_lens)
print()

# 모델이 최대 출력할 수 있는 토큰수와 비교해서 넘어가는지 아닌지 확인
n_too_long = sum(i > max_output_tokens for i in total_tokens_lens)
print(f'\n {n_too_long}개의 대화가 {max_output_tokens}개 토큰 제한을 초과합니다')

system 메시지 누락 수 0
user 메시지 누락 수 0

### 대화당 메시지 수 통계:
min / max: 3, 3
mean / median: 3.0, 3.0

### 대화당 전체 토큰 수 통계:
min / max: 69, 110
mean / median: 85.9, 85.5

### 대화당 모델 출력 토큰 수 통계:
min / max: 22, 51
mean / median: 34.5, 33.0


 0개의 대화가 16384개 토큰 제한을 초과합니다


In [16]:
dataset

[{'messages': [{'role': 'system', 'content': '너는 지식이 풍부하지만 시크하게 응답하는 챗봇이야.'},
   {'role': 'user', 'content': "제주도는 '삼다도'라고 불리기도 하는데 '삼다'에 해당되는 것은 무엇일까요?"},
   {'role': 'assistant',
    'content': "정답은 '여자', '바람', '돌' 이고 이 세가지가 많다고 해서 '삼다' 라고 불림."}]},
 {'messages': [{'role': 'system', 'content': '너는 지식이 풍부하지만 시크하게 응답하는 챗봇이야.'},
   {'role': 'user',
    'content': '뮤지컬, 연극, 오페라, 음악회 등의 공연이 끝난 후에 관객이 박수를 보내 배우들을 다시 무대로 나오게 하는 것을 무엇이라 할까요?'},
   {'role': 'assistant',
    'content': "정답은 '커튼콜'이며 '앵콜'과는 달리 추가 무대를 요청하는 것이 아니라 단순히 등장을 요구하는 것."}]},
 {'messages': [{'role': 'system', 'content': '너는 지식이 풍부하지만 시크하게 응답하는 챗봇이야.'},
   {'role': 'user', 'content': '프랑스의 수도는 어디인가요?'},
   {'role': 'assistant',
    'content': '프랑스의 수도는 파리임. 파리는 예술의 도시라고 불리며 세계 최고의 관광 도시 중 하나임.'}]},
 {'messages': [{'role': 'system', 'content': '너는 지식이 풍부하지만 시크하게 응답하는 챗봇이야.'},
   {'role': 'user', 'content': '세계에서 가장 큰 바다는 무엇인가요?'},
   {'role': 'assistant',
    'content': '세계에서 가장 큰 바다는 태평양임. 태평양은 오대양의 하나로 지구 표면의 1/3을 차지하며 표면적

#### 3) 적정 epochs 및 비용 추정

In [17]:
MAX_TOKENS_PER_EXAMPLE = 16384   # gpt-4o-mini 기준 최대 출력 토큰 수

# fine-tuning에서 성능 향상을 보기 위해 필요한 적정 epochs를 지정하는 단계
# -> 데이터의 양과 과대적합을 고려하여 OpenAI의 테스트에 얻어진 적정 기준
TARGET_EPOCHS = 3                    # 원하는 기본 epochs
MIN_DEFAULT_EPOCHS = 1            # openAI가 제시하는 fine-tuning에 필요한 최소 epochs
MAX_DEFAULT_EPOCHS = 25           # 최대 epochs (이 횟수 초과하면 과대적합 가능성 ^)
MIN_TARGET_EXAMPLES = 100         # openAI가 제시하는 최소 데이터 개수
MAX_TARGET_EXAMPLES = 25000       # 최대 데이터 개수 (시간 및 비용 효율을 위해 상한 설정)

# 데이터의 양과 사용자가 원하는 epoch 수에 따라 최저기준과 최대기준을 비교하여 적정 epoch 구하게 됨


n_epochs = TARGET_EPOCHS          # 기준 타겟 epoch에 따라 실제로 적용될 epoch
n_train_examples = len(dataset)    # 대화의 개수(데이터 개수)

# 타겟 epoch와 우리 데이터 개수를 곱한 값이 OpenAI의 최소 기준 샘플수보다 작다면
# OpenAI 기준 epoch 최대치인 25회와 최소 데이터 100개에서 우리 데이터를 나눈 몫을 비교하여
# 더 낮은 값으로 최종 epoch를 설정하겠다는 뜻
if (n_train_examples * TARGET_EPOCHS) < MIN_TARGET_EXAMPLES :     
    n_epochs = min(MAX_DEFAULT_EPOCHS, (MIN_TARGET_EXAMPLES // n_train_examples))

elif (n_train_examples * TARGET_EPOCHS) > MAX_TARGET_EXAMPLES :   
    n_epochs = max(MIN_DEFAULT_EPOCHS, (MAX_TARGET_EXAMPLES // n_train_examples))

# 각 대화별 총 토큰수 값들을 가져와서 16384개와 비교해 더 낮은 값으로 총 토큰수 계산
# (어차피 더 길게 넣어도 16384개까지만 요금이 청구되니까 정밀하게 계산)
n_billing_tokens = sum(min(MAX_TOKENS_PER_EXAMPLE, i) for i in total_tokens_lens)

print(f'해당 데이터셋에는 학습 중 요금이 청구될 {n_billing_tokens}개의 토큰이 있습니다.')
print(f'기본적으로 이 데이터셋에서 {n_epochs} epoch 동안 학습합니다.')
print(f'총 {n_epochs * n_billing_tokens} 개의 토큰에 대해 요금이 청구됩니다.')

해당 데이터셋에는 학습 중 요금이 청구될 859개의 토큰이 있습니다.
기본적으로 이 데이터셋에서 10 epoch 동안 학습합니다.
총 8590 개의 토큰에 대해 요금이 청구됩니다.


## **2. 학습 데이터셋 업로드 파일 객체 생성**

In [ ]:
# fine-tuning API를 사용하기 위한 파일 객체 만들기
fine_tune_files = client.files.create(
    file=open('data/common_sense_train.jsonl', 'rb'),
    purpose='fine-tune'
)

fine_tune_files

FileObject(id='file-MdfbnhHrjMHrRkPx9XcBkA', bytes=6176, created_at=1774939720, filename='common_sense_train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

## **3. Fine-tuning 작업 진행 및 결과 확인**
- 파인튜닝 작업이 완료된 모델은 OpenAI 서버(홈페이지 확인 가능)에 저장되며 작업에 할당된 모델 ID값을 통해 액세스 할 수 있음
- 작업이 완료되지 않거나, fine-tuning중 잘못된 부분이 생각나 작업을 취소해도 요금 부과 안 됨

In [25]:
# 작업 객체 생성 및 fine-tuning 시작!

fine_tune_job = client.fine_tuning.jobs.create(
    model='gpt-4o-mini-2024-07-18',
    # 위에서 만든 파일 업로드 객체의 id값 입력
    training_file = fine_tune_files.id
)

fine_tune_job

FineTuningJob(id='ftjob-PhqHVkEj4SVZ8RxRWRAgBOvY', created_at=1774939902, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-NIS2YRUXpw47qQ31hSVYDspr', result_files=[], seed=1487251306, status='validating_files', trained_tokens=None, training_file='file-MdfbnhHrjMHrRkPx9XcBkA', validation_file=None, estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'))), user_provided_suffix=None, usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_experiment_id=None)

#### 1) 작업 ID 가져오기

In [24]:
fine_tune_job.id

'ftjob-jBCxtIhLKzWHoTr4oYMx6BFF'

#### 2) 작업 ID에 따른 개별 작업 세부사항 확인
- retrieve (개별 조회)
- 대상: 내가 방금 실행한 특정한 하나의 학습 작업(Job ID).

In [59]:
client.fine_tuning.jobs.retrieve(fine_tune_job.id)

# fine_tuned_model은 파인튜닝 완료 후 생성된 개인 모델명
# status는 validating_file(파일검증), running(동작 중), queued(대기), succeeded(성공), failed(실패), cancelled(취소)

FineTuningJob(id='ftjob-PhqHVkEj4SVZ8RxRWRAgBOvY', created_at=1774939902, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4o-mini-2024-07-18:fininsight::DPNJcMEg', finished_at=1774940195, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=1.8, n_epochs=10), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-NIS2YRUXpw47qQ31hSVYDspr', result_files=['file-T3uUXkwuUo27FUucKSoURb'], seed=1487251306, status='succeeded', trained_tokens=8390, training_file='file-MdfbnhHrjMHrRkPx9XcBkA', validation_file=None, estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=1.8, n_epochs=10))), user_provided_suffix=None, usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_

In [26]:
# 실시간 fine-tuning 진행사항 확인 코드
while True :
    fine_tune_job_info = client.fine_tuning.jobs.retrieve(fine_tune_job.id)
    print(f'진행 상태: {fine_tune_job_info.status}')

    if fine_tune_job_info.status=='succeeded' :
        print('파인튜닝 완료')
        break
    elif fine_tune_job_info.status=='failed' :
        print('파인튜닝 실패')
        break
    time.sleep(30)  # 30초 간격으로 체크

진행 상태: queued
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: running
진행 상태: succeeded
파인튜닝 완료


In [ ]:
# 모델명 확인
fine_tune_job_info.fine_tuned_model

'ft:gpt-4o-mini-2024-07-18:fininsight::DPNJcMEg'

#### 3) 로그, 성능 업데이트 또는 오류메시지 같은 미세조정 작업 중 발생한 이벤트를 추적
- 특정 작업의 타임라인
- Loss(손실값) 변화, 상세 에러 확인

In [28]:
# 작업시작, 전처리, 체크포인트(중간 모델 저장), 검증, 완료, 실패 등....
# 모든 세부 사건에 대한 로그 확인
client.fine_tuning.jobs.list_events(fine_tune_job.id, limit=3)  # limit으로 개수 제한

SyncCursorPage[FineTuningJobEvent](data=[FineTuningJobEvent(id='ftevent-aXsdiC6phy5fBr0r4LzMS9sV', created_at=1774940923, level='info', message='The job has successfully completed', object='fine_tuning.job.event', data={}, type='message'), FineTuningJobEvent(id='ftevent-olYn2V3m1YyYFolYAv4iVouQ', created_at=1774940921, level='info', message='Usage policy evaluations completed, model is now enabled for sampling', object='fine_tuning.job.event', data={}, type='message'), FineTuningJobEvent(id='ftevent-Zsx82f0Smh8R7lcph7jQDcAt', created_at=1774940921, level='info', message='Moderation checks for snapshot ft:gpt-4o-mini-2024-07-18:fininsight::DPNJcMEg passed.', object='fine_tuning.job.event', data={'blocked': False, 'results': [{'flagged': False, 'category': 'harassment/threatening', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'sexual', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'sexual/minors', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'prop

#### 4) fine-tuning 전체 작업 목록 출력
- list (전체 목록)
- 대상: 내 계정에서 실행했던 모든 파인튜닝 작업들의 리스트.

In [ ]:
jobs = client.fine_tuning.jobs.list(limit=3)
# for job in jobs :
#     print(f'작업ID {job.id}, 결과모델: {job.fine_tuned_model}')

SyncCursorPage[FineTuningJob](data=[FineTuningJob(id='ftjob-lUOkzkiOrTBb392TcU9qbiOC', created_at=1774944061, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4o-mini-2024-07-18:fininsight::DPOSq7DK', finished_at=1774944611, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=0.8, n_epochs=5), model='ft:gpt-4o-mini-2024-07-18:fininsight::DPMIyYTG', object='fine_tuning.job', organization_id='org-NIS2YRUXpw47qQ31hSVYDspr', result_files=['file-8Cb4KBk75P8A4n5wMBCR19'], seed=479802209, status='succeeded', trained_tokens=3530, training_file='file-16D3fvXjwsqeN4KC67a2ai', validation_file='file-TTVoYNxxLq4dRZHEetX3Ey', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=0.8, n_epochs=5))), user_provided_suffix=None, usage_metrics=None, shared_with_openai=False, eval_id=None, i

#### 5) 작업 취소

In [ ]:
# 작업 취소
# client.fine_tuning.jobs.cancel('ftjob-y8AJuUPqpQ5poPoVi03zEeqF')

#### 6) 완성된 fine-tuning 모델 삭제

In [31]:
# client.models.delete('완성된 모델명')  # id값이 아닌 '완성된 모델명'이어야 함

#### Fine-tuning 완료된 작업 검색 (python코드를 사용하지 않는 수동 방식)
- 계층구조를 사람이 보기 쉽게 출력 가능
- **!curl** : 다양한 프로토콜을 사용해 데이터를 전송하는데 널리 사용되는 명령줄 도구로 API와 직접 상호작용하는데 유용함
- **-H** : HTTP 요청에 사용자 정의 header를 추가하는 옵션으로 사용자 인증이나 컨텐츠 타입을 지정할 때 사용

In [49]:
!curl https://api.openai.com/v1/fine_tuning/jobs/ftjob-jBCxtIhLKzWHoTr4oYMx6BFF \
-H "Authorization: Bearer $MY_API_KEY"

{
  "object": "fine_tuning.job",
  "id": "ftjob-jBCxtIhLKzWHoTr4oYMx6BFF",
  "model": "gpt-4o-mini-2024-07-18",
  "created_at": 1774939722,
  "finished_at": 1774939797,
  "fine_tuned_model": null,
  "organization_id": "org-NIS2YRUXpw47qQ31hSVYDspr",
  "result_files": [],
  "status": "failed",
  "validation_file": null,
  "training_file": "file-MdfbnhHrjMHrRkPx9XcBkA",
  "hyperparameters": {
    "n_epochs": 10,
    "batch_size": 1,
    "learning_rate_multiplier": 1.8
  },
  "trained_tokens": null,
  "error": {
    "code": "server_error",
    "param": null,
    "message": "We're having trouble accessing your files right now. Please try again later."
  },
  "user_provided_suffix": null,
  "seed": 289795317,
  "estimated_finish": null,
  "integrations": [],
  "metadata": null,
  "usage_metrics": null,
  "shared_with_openai": false,
  "eval_id": null,
  "internal_worker_backend": null,
  "internal_peashooter_execution": null,
  "train_experiment_id": null,
  "eval_experiment_id": null,
  "m

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  1191  100  1191    0     0   3580      0 --:--:-- --:--:-- --:--:--  3631


### 4. Fine-tuning 완료 후 질의 응답

In [39]:
completions = client.chat.completions.create(
    model='ft:gpt-4o-mini-2024-07-18:fininsight::DPNJcMEg',
    messages=[{'role':'user', 'content':'우리나라의 첫 대통령은 누구야?'}]
)
print(completions.choices[0].message.content)

우리나라의 첫 대통령은 이승만이며 무소속 출신 이었음.


>#### **fine-tuning 결과가 좋지 않다고 판단될 경우 대응법**
>#### 1. 데이터 부분
>1) 데이터 수 추가
>    - 가장 효과 확실
>2) 데이터의 균형과 다양성 고려
>    - 균형을 맞춰야 한다면 적은 양의 고품질 데이터가, 많은 양의 저품질 데이터보다 효과적
>3) 기존 데이터의 문제점 조사
>    - 원치않는 질의-응답 방식의 대화 데이터가 포함되어 있는지 확인 (잘못된 패턴이 학습될 수 있음)
>    - 응답에 필요한 모든 정보가 포함되어 있는지 확인 (할루시네이션 발생 가능성 존재)
>    - 여러 사람이 만든 데이터를 합쳤다면 일관성 확인 (패턴이 너무 중구난방이면 학습하기 힘듦)
>
>#### 2. 모델 부분
>1) 완료된 모델에 추가로 fine-tuning 진행
>    - 학습이 부족했을 수 있으므로 추가 진행
>2) 하이퍼파라미터 변경
>    - hyperparameters={'n_epochs':}와 같이 학습시 파라미터 커스텀 가능

### **5. 체크포인트 (중간저장) 모델 활용하기**
- fine-tuning 완료시 자동으로 가장 최근 3개의 체크포인트 모델을 활용할 수 있음
- 최종 모델의 성능이 애매하거나, 과대적합이라고 판단되면 이전 체크포인트 모델을 사용할 수 있음

#### 1) 체크포인트 모델 확인

In [ ]:
!curl https://api.openai.com/v1/fine_tuning/jobs/ftjob-~~~/checkpoints \
-H "Authorization: Bearer $MY_API_KEY"

#### 2) 체크포인트 모델별 평가 지표 확인
- loss : 시퀀스(토큰)에 대한 Cross Entropy Loss
모델이 예측한 토큰의 확률분포와 실제 정답 토큰(원핫벡터) 간의 오차를 계산

$$Loss = -\frac{1}{N} \sum_{i=1}^{N} \frac{1}{M_i} \sum_{j=1}^{M_i} y_{i,j} \log(\hat{y}_{i,j})$$

* $N$: 전체 데이터(대화)의 개수
* $M_i$: $i$번째 데이터의 총 토큰 수
* $y_{i,j}$: $i$번째 데이터의 $j$번째 실제 정답 토큰 (정답일 경우 1, 아닐 경우 0의 값을 가지는 원핫 인코딩 벡터)
* $\hat{y}_{i,j}$: $i$번째 데이터의 $j$번째 토큰에 대해 모델이 예측한 확률 (Softmax가 적용된 0~1 사이의 실수 값)

#### Mean token accuracy : 평균 토큰 정확도
실제 문장의 토큰과 모델이 예측한 토큰을 비교하여, 올바르게 예측된 코튼의 비율을 구한 뒤 전체 데이터에 대해 평균을 낸 값

$$Accuracy = \frac{1}{N} \sum_{i=1}^{N} \frac{\text{Number of Correct Tokens}_i}{\text{Total Tokens in Sample}_i}$$

* $\text{Number of Correct Tokens}_i$: $i$번째 데이터에서 모델이 정확하게 예측한 토큰의 수
* $\text{Total Tokens in Sample}_i$: $i$번째 데이터의 전체 토큰 수

90번

In [ ]:
completions = client.chat.completions.create(
    model='ft:gpt-4o-mini-2024-07-18:personal::DPM75rwT:ckpt-step-90',
    messages=[{'role':'user', 'content':'우리나라의 첫 대통령은 누구인가?'}]
)
print(completions.choices[0].message.content)

## Fine-tuning된 모델에 추가 fine-tuning 진행하기
- 추가로 학습할 데이터 로드
- 모델은 fine-tune 완료된 모델명
- 검증용 데이터셋을 추가하고 하이퍼파라미터도 변경해보자

In [44]:
with open('data/common_sense_train_add.jsonl', 'r', encoding='utf-8') as f :
    dataset = [json.loads(line) for line in f]

print(len(dataset))
dataset

10


[{'messages': [{'role': 'system', 'content': '너는 지식이 풍부하지만 시크하게 응답하는 챗봇이야.'},
   {'role': 'user', 'content': '화가 바뀌어 오히려 복이 된다 라는 뜻의 사자성어는 무엇인가요?'},
   {'role': 'assistant', 'content': "'전화위복'"}]},
 {'messages': [{'role': 'system', 'content': '너는 지식이 풍부하지만 시크하게 응답하는 챗봇이야.'},
   {'role': 'user', 'content': '새끼 손가락을 지칭하는 명칭은 무엇인가요?'},
   {'role': 'assistant', 'content': "새끼 손가락은 '약지'이고 추가로 첫번째 손가락은 '엄지'"}]},
 {'messages': [{'role': 'system', 'content': '너는 지식이 풍부하지만 시크하게 응답하는 챗봇이야.'},
   {'role': 'user', 'content': '물이 끓는 온도는 몇 도인가요?'},
   {'role': 'assistant', 'content': '물은 100도에서 끓음'}]},
 {'messages': [{'role': 'system', 'content': '너는 지식이 풍부하지만 시크하게 응답하는 챗봇이야.'},
   {'role': 'user', 'content': '버스가 갑자기 정지하면 우리 몸이 앞으로 쏠리는 현상을 무엇이라고 하나요?'},
   {'role': 'assistant',
    'content': "'관성의 법칙'이라고 하며 가만히 있는 물체는 계속 가만히 있고, 움직이는 물체는 계속 움직이려는 현상을 뜻함"}]},
 {'messages': [{'role': 'system', 'content': '너는 지식이 풍부하지만 시크하게 응답하는 챗봇이야.'},
   {'role': 'user', 'content': '식물이 자라는 데 필요한 세 가지 요소는 무엇인가요?'}

In [ ]:
# 추가 학습용 데이터 10개~ 파일 객체
fine_tune_files_add = client.files.create(
    file=open('data/common_sense_train_add.jsonl', 'rb'),
    purpose='fine-tune'
)

fine_tune_files_add

FileObject(id='file-2KRjdFjwKtR8MVRSKK1JhS', bytes=5167, created_at=1774942967, filename='common_sense_train_add.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [46]:
# 검증용 데이터 5개
fine_tune_files_val = client.files.create(
    file=open('data/common_sense_val.jsonl', 'rb'),
    purpose='fine-tune'
)

fine_tune_files_val

FileObject(id='file-45T3g4bqLWPhdKuRU9AYS6', bytes=2513, created_at=1774943157, filename='common_sense_val.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [ ]:
# 추가 fine-tuning 진행
fine_tune_job_add = client.fine_tuning.jobs.create(
    model='ft:gpt-4o-mini-2024-07-18:fininsight::DPNJcMEg',
    training_file=fine_tune_files_add.id,     # 추가할 학습 데이터
    validation_file=fine_tune_files_val.id     # 검증할 검증 데이터
    , hyperparameters={'n_epochs': 5,
                       'learning_rate_multiplier': 0.8
                       }
)

# <대표적인 하이퍼파라미터 종류>
 # n_epochs : 학습 횟수
 # batch_size : 배치 사이즈
 # learning_rate_multiplier : 학습률 배수 (1은 기본 lr 그대로 사용, 0.5sms 기본 lr의 50% 사용)
 # weight_decay : 가중치 감소(%로 줄여주는 L2 규제) - 필요시 0.1~0.001 사이로 사용 (규제 강도)

fine_tune_job_add

FineTuningJob(id='ftjob-HVum0mV8Vr11NP2BJljRelp6', created_at=1774943448, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier=0.8, n_epochs=5), model='ft:gpt-4o-mini-2024-07-18:fininsight::DPNJcMEg', object='fine_tuning.job', organization_id='org-NIS2YRUXpw47qQ31hSVYDspr', result_files=[], seed=544919091, status='validating_files', trained_tokens=None, training_file='file-2KRjdFjwKtR8MVRSKK1JhS', validation_file='file-45T3g4bqLWPhdKuRU9AYS6', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size='auto', learning_rate_multiplier=0.8, n_epochs=5))), user_provided_suffix=None, usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_ex

In [54]:
!curl https://api.openai.com/v1/fine_tuning/jobs/ftjob-HVum0mV8Vr11NP2BJljRelp6/ \
-H "Authorization: Bearer $MY_API_KEY"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0


In [55]:
!curl https://api.openai.com/v1/fine_tuning/jobs/ftjob-HVum0mV8Vr11NP2BJljRelp6/checkpoints \
-H "Authorization: Bearer $MY_API_KEY"

{
  "object": "list",
  "data": [
    {
      "object": "fine_tuning.job.checkpoint",
      "id": "ftckpt_4atf13JfiPeLLmCV6N7N4LmL",
      "created_at": 1774943933,
      "fine_tuned_model_checkpoint": "ft:gpt-4o-mini-2024-07-18:fininsight::DPOISm97",
      "fine_tuning_job_id": "ftjob-HVum0mV8Vr11NP2BJljRelp6",
      "metrics": {
        "step": 50,
        "train_loss": 0.0029888153076171875,
        "valid_loss": 1.722958700997489,
        "full_valid_loss": 0.8898065604415595,
        "train_mean_token_accuracy": 1.0,
        "valid_mean_token_accuracy": 0.6428571428571429,
        "full_valid_mean_token_accuracy": 0.8431372549019608,
        "type": "supervised"
      },
      "step_number": 50,
      "storage_tier": null,
      "latest_export_request": null
    },
    {
      "object": "fine_tuning.job.checkpoint",
      "id": "ftckpt_eY6gKiAliAtZc74vKAUVoBlA",
      "created_at": 1774943871,
      "fine_tuned_model_checkpoint": "ft:gpt-4o-mini-2024-07-18:fininsight::DPOIS4PR:ckp

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  2450  100  2450    0     0   8464      0 --:--:-- --:--:-- --:--:--  8536


In [53]:
res = client.fine_tuning.jobs.list_events(fine_tune_job_add.id, limit=5)
print(res.model_dump_json(indent=2))    # JSON형식으로 계층은 space 2칸씩 지정

{
  "data": [
    {
      "id": "ftevent-gNAWXn33sRbRAOJrE7jyXFGH",
      "created_at": 1774944695,
      "level": "info",
      "message": "The job has successfully completed",
      "object": "fine_tuning.job.event",
      "data": {},
      "type": "message"
    },
    {
      "id": "ftevent-LqxaiIkd65iInCnYGJZUYLKP",
      "created_at": 1774944693,
      "level": "info",
      "message": "Usage policy evaluations completed, model is now enabled for sampling",
      "object": "fine_tuning.job.event",
      "data": {},
      "type": "message"
    },
    {
      "id": "ftevent-mIGzalZZcK1rDIegjhmWLyfT",
      "created_at": 1774944693,
      "level": "info",
      "message": "Moderation checks for snapshot ft:gpt-4o-mini-2024-07-18:fininsight::DPOISm97 passed.",
      "object": "fine_tuning.job.event",
      "data": {
        "blocked": false,
        "results": [
          {
            "flagged": false,
            "category": "harassment/threatening",
            "enforcement": "bloc